# 03 News NLP Preprocessing

This notebook executes Phase 4 end-to-end using `src.nlp.cleaner.clean_news` and `src.nlp.sentiment.run_finbert`.
It creates cleaned text, metadata, train-only TF-IDF artifacts, sentiment aggregates, and lagged daily news features.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import yaml

root = Path.cwd()
if not (root / 'pipeline_config.yaml').exists() and (root.parent / 'pipeline_config.yaml').exists():
    root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Working directory: {root}')

In [ ]:
from src.nlp.cleaner import clean_news
from src.nlp.sentiment import run_finbert

with open('pipeline_config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

clean_news(config)
run_finbert(config)
print('Phase 4 NLP features generated.')

In [ ]:
import pandas as pd

news_features_path = Path(config['paths']['news_features'])
df = pd.read_parquet(news_features_path)

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Date range:', df['date'].min(), '->', df['date'].max())
df.head()

In [ ]:
required_artifacts = [
    'datasets/features/news_features.parquet',
    'reports/nlp_phase4_diagnostics.md',
    'reports/nlp_tfidf_top_terms.md',
    'reports/plots/nlp_embedding_umap_sample.png',
]

artifact_df = pd.DataFrame({'artifact': required_artifacts})
artifact_df['exists'] = artifact_df['artifact'].map(lambda p: Path(p).exists())
artifact_df